In [ ]:
import pandas as pd
import time
from google import genai #Check API usage & rate limits at: https://aistudio.google.com/projects
from google.genai import types
import CJDH_local_settings

In [19]:
client = genai.Client(api_key=CJDH_local_settings.local_settings['GenAI_Settings']['google_genai_API_KEY'])
MODEL_ID = "gemini-2.5-flash-lite"

df = pd.read_csv('2526players_mids.csv')

def classify_batch_minimal(player_names):
    names_input = ",".join(player_names)

    prompt = f"Classify each midfielder as Defensive (D), Attacking (A) or Unclear (U). Output only: D,A,U...\n{names_input}"

    for attempt in range(3):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=2001
            )
        )
        
        labels = response.text.strip().replace(" ", "").split(",")
        
        if len(labels) == len(player_names) and all(l in ['D','A','U'] for l in labels):
            return labels
        
        if attempt < 2:
            time.sleep(1)
    
    print(f"Failed batch of {len(player_names)}, marking as U")
    return ['U'] * len(player_names)

# Execution
batch_size = 1000
all_labels = []
player_list = df['player_name_id'].tolist()

for i in range(0, len(player_list), batch_size):
    batch = player_list[i:i+batch_size]
    print(f"Batch {i//batch_size + 1}: {len(batch)} players")
    labels = classify_batch_minimal(batch)
    all_labels.extend(labels)
    time.sleep(4)

df['position_code'] = all_labels

Batch 1: 356 players
Failed batch of 356, marking as U


In [17]:
df['position_code'].value_counts()

position_code
U    356
Name: count, dtype: int64

In [18]:
df[df['position_code'] == 'D'].head()

,player_name_id,team_name,position_code
